# Part C — Native agent evaluation
## Notebook 04

This notebook closes the **Observability** pillar: server-side traces (gtm-03) show you *what the agent did*;
native evaluation tells you *whether it did the right thing* — and both are impossible when the reasoning loop
lives outside Snowflake. A migration is only trustworthy if the **agent itself** is measured, not a raw model
call. Here we run Snowflake's **native Cortex Agent Evaluation** against `GTM_SUPERVISOR`, the supervisor that
routes to the three specialists built in Part B (Scoring, Recommendation, Coaching).

We follow the **GPA** framing (Snowflake AI Research):

| Dimension | Question | Metric |
|---|---|---|
| **G**oal | Did the agent produce the right answer? | `answer_correctness` |
| **P**lan / **A**ction | Did it route to the right specialist and reason coherently? | `tool_selection_accuracy`, `logical_consistency` |
| Custom | Is routing optimal for the query intent? | `routing_quality` (LLM-as-judge) |

The loop: register a labeled dataset → run a **baseline** → inspect by intent → **improve** the supervisor's
orchestration → re-run → **compare** → **promote** the better version to a `production` alias → wire up
**regression** detection. Ground truth lives in `AGENT_EVAL_QUESTIONS` (created by `setup.sql`).

**Prerequisite:** `gtm-03-after-agents` completed (the supervisor + specialists exist).

> **Prereq note:** the LLM judge uses cross-region inference — it must be enabled on the account. Evaluations
> also run via a managed task, so the evaluating role needs `EXECUTE TASK ON ACCOUNT` (granted in Section 1).

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session
import time, json, yaml, tempfile, os
session = get_active_session()
for s in ['USE ROLE SYSADMIN','USE DATABASE GTMAGENTS','USE SCHEMA GTMAGENTS.DEMO','USE WAREHOUSE GTMAGENTS_WH']:
    session.sql(s).collect()
# Reading AI Observability + feedback for the agent needs MONITOR on it (idempotent).
session.sql('GRANT MONITOR ON AGENT GTMAGENTS.DEMO.GTM_SUPERVISOR TO ROLE GTMAGENTS_ROLE').collect()
# Agent evaluations run via a MANAGED TASK, so the evaluating role needs EXECUTE TASK ON ACCOUNT.
# This grant requires ACCOUNTADMIN; run it once (safe to skip if already granted).
try:
    session.sql('USE ROLE ACCOUNTADMIN').collect()
    session.sql('GRANT EXECUTE TASK ON ACCOUNT TO ROLE SYSADMIN').collect()
    session.sql('GRANT EXECUTE MANAGED TASK ON ACCOUNT TO ROLE SYSADMIN').collect()
    print('EXECUTE TASK granted to SYSADMIN.')
except Exception as e:
    print(f'Skipping EXECUTE TASK grant (needs ACCOUNTADMIN): {e}')
finally:
    session.sql('USE ROLE SYSADMIN').collect()
print('Connected to GTMAGENTS.DEMO')

---
## Section 2: The labeled evaluation dataset

`AGENT_EVAL_QUESTIONS` (from `setup.sql`) holds 12 questions spanning the full intent distribution. Each
`GROUND_TRUTH` row carries `ground_truth_output` (what a correct answer looks like), `ground_truth_invocations`
(the specialist tool(s) the supervisor **should** call), plus `intent` and `process` labels so we can slice
scores by scenario — including the **high-risk** refusal cases the agent must decline.

In [ ]:
session.sql("""
    SELECT ground_truth:process::string AS process,
           ground_truth:intent::string  AS intent,
           COUNT(*) AS n
    FROM AGENT_EVAL_QUESTIONS GROUP BY 1,2 ORDER BY 1,2
""").show()

---
## Section 3: Register the evaluation dataset

`SYSTEM$CREATE_EVALUATION_DATASET` turns the table into a first-class dataset the evaluation engine consumes.
`query_text` maps to the user question; `expected_tools` maps to the ground-truth VARIANT.

In [ ]:
session.sql('DROP DATASET IF EXISTS GTMAGENTS.DEMO.GTM_EVAL_DATASET').collect()
session.sql("""
    CALL SYSTEM$CREATE_EVALUATION_DATASET(
      'Cortex Agent',
      'GTMAGENTS.DEMO.AGENT_EVAL_QUESTIONS',
      'GTMAGENTS.DEMO.GTM_EVAL_DATASET',
      OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH')
    )
""").collect()
print('Dataset registered: GTM_EVAL_DATASET')
session.sql('SHOW DATASETS IN SCHEMA GTMAGENTS.DEMO').show()

---
## Section 4: Write + upload the evaluation config

The config names the agent under test, the dataset, and the **metrics**. We use three built-ins plus one custom
LLM-as-judge metric (`routing_quality`) that scores whether the supervisor picked the optimal specialist for the
query intent — the highest-leverage thing to get right in a routing agent.

In [ ]:
eval_config_yaml = """
evaluation:
  agent_params:
    agent_name: "GTMAGENTS.DEMO.GTM_SUPERVISOR"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "GTM Supervisor routing + answer evaluation"
  source_metadata:
    type: "dataset"
    dataset_name: "GTMAGENTS.DEMO.GTM_EVAL_DATASET"

metrics:
  - "answer_correctness"
  - "tool_selection_accuracy"
  - "logical_consistency"
  - name: "routing_quality"
    score_ranges:
      min_score: [1, 3]
      median_score: [4, 6]
      max_score: [7, 10]
    prompt: |
      Evaluate whether the GTM supervisor routed the query to the correct specialist(s).

      User query: {{input}}
      Tools used: {{tool_info}}
      Expected behavior: {{ground_truth}}
      Agent response: {{output}}

      The specialists are: ScoringSpecialist (score one email's buyer intent by id),
      RecommendationSpecialist (GTM performance metrics and winning patterns),
      CoachingSpecialist (rewrite a weak email to the framework).

      Rate 1-10:
      1-3  = Wrong specialist, missing a needed specialist, or called a tool on an out-of-scope question.
      4-6  = Right primary specialist but missed a secondary one on a compound request, or made a redundant call.
      7-10 = Optimal routing: correct specialist(s) for the intent, and NO tool call on out-of-scope questions.

      Provide your score as a single integer.
""".strip()

tmp_path = os.path.join(tempfile.gettempdir(), 'eval_config.yaml')
with open(tmp_path, 'w') as f:
    f.write(eval_config_yaml)
# Sanity-check it is valid YAML before uploading.
yaml.safe_load(eval_config_yaml)
session.sql(f"PUT 'file://{tmp_path}' @GTMAGENTS.DEMO.EVAL_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()
os.unlink(tmp_path)
print('Uploaded @GTMAGENTS.DEMO.EVAL_STAGE/eval_config.yaml')
session.sql('LIST @GTMAGENTS.DEMO.EVAL_STAGE').show()

---
## Section 5: Run the baseline evaluation

`EXECUTE_AI_EVALUATION('START', ...)` launches the run: it replays every question through `GTM_SUPERVISOR`,
captures the trace, and scores each metric with the LLM judge. This takes a few minutes — poll `STATUS`.

In [ ]:
session.sql("""
    CALL EXECUTE_AI_EVALUATION(
      'START',
      OBJECT_CONSTRUCT('run_name', 'baseline-v1'),
      '@GTMAGENTS.DEMO.EVAL_STAGE/eval_config.yaml'
    )
""").collect()
print("Evaluation 'baseline-v1' started — poll the next cell until COMPLETED.")

In [ ]:
def poll_eval(run_name, max_iter=40, wait_s=30):
    # Terminal states only — note INVOCATION_COMPLETED is an intermediate state, not the end.
    terminal = {'COMPLETED', 'PARTIALLY_COMPLETED', 'CANCELLED', 'FAILED'}
    for i in range(max_iter):
        status = session.sql(f"""
            CALL EXECUTE_AI_EVALUATION('STATUS',
              OBJECT_CONSTRUCT('run_name', '{run_name}'),
              '@GTMAGENTS.DEMO.EVAL_STAGE/eval_config.yaml')
        """).collect()
        s = str(status[0][0]).upper()
        print(f'[{i+1}/{max_iter}] {run_name}: {s}')
        if s in terminal or 'FAILED' in s:
            return s
        time.sleep(wait_s)
    return 'still running — re-run this cell'

print(poll_eval('baseline-v1'))

---
## Section 6: Inspect the baseline

`GET_AI_EVALUATION_DATA` returns one row per (question x metric). We look at overall averages, the lowest-scoring
questions with the judge's explanation, then a pivot by **intent x process** to see exactly where the agent is weak.

In [ ]:
session.sql("""
    SELECT METRIC_NAME, METRIC_TYPE, COUNT(*) AS n,
           ROUND(AVG(EVAL_AGG_SCORE),3) AS avg_score,
           ROUND(MIN(EVAL_AGG_SCORE),3) AS min_score,
           ROUND(MAX(EVAL_AGG_SCORE),3) AS max_score
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','baseline-v1'))
    GROUP BY METRIC_NAME, METRIC_TYPE ORDER BY avg_score ASC
""").show()

In [ ]:
session.sql("""
    SELECT INPUT, METRIC_NAME, ROUND(EVAL_AGG_SCORE,3) AS score,
           METRIC_CALLS[0]:explanation::varchar AS explanation
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','baseline-v1'))
    ORDER BY EVAL_AGG_SCORE ASC LIMIT 8
""").show(max_width=120)

In [ ]:
session.sql("""
    SELECT e.GROUND_TRUTH:process::varchar AS process,
           e.GROUND_TRUTH:intent::varchar  AS intent,
           ROUND(AVG(CASE WHEN r.METRIC_NAME='answer_correctness'      THEN r.EVAL_AGG_SCORE END),3) AS answer_correctness,
           ROUND(AVG(CASE WHEN r.METRIC_NAME='tool_selection_accuracy' THEN r.EVAL_AGG_SCORE END),3) AS tool_selection,
           ROUND(AVG(CASE WHEN r.METRIC_NAME='logical_consistency'     THEN r.EVAL_AGG_SCORE END),3) AS logical_consistency,
           ROUND(AVG(CASE WHEN r.METRIC_NAME='routing_quality'         THEN r.EVAL_AGG_SCORE END),3) AS routing_quality
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','baseline-v1')) r
    JOIN AGENT_EVAL_QUESTIONS e ON r.INPUT = e.INPUT_QUERY
    GROUP BY 1,2 ORDER BY 1,2
""").show(max_width=150)

---
## Section 7: Iterate — improve the supervisor's orchestration

The highest-leverage fix for a routing agent is the **orchestration instructions**. We make the routing rules
explicit — including a hard rule to **decline out-of-scope questions without calling any tool**, and to call
**multiple** specialists for compound requests. `ALTER AGENT ... MODIFY LIVE VERSION SET SPECIFICATION` edits the
working version in place.

In [ ]:
session.sql(r"""
ALTER AGENT GTM_SUPERVISOR MODIFY LIVE VERSION SET SPECIFICATION = $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 60
    tokens: 24000
instructions:
  response: "Synthesize specialist outputs into one concise, actionable answer for a sales manager. If you declined a request, briefly say what you CAN help with."
  orchestration: |
    ROUTING RULES (follow strictly):
    1. SCORING an email's buyer intent (mentions an email id or 'intent score') -> ScoringSpecialist.
    2. PERFORMANCE questions (reply/meeting/win rates, revenue, which region/team/tier/month wins) -> RecommendationSpecialist.
    3. REWRITE / IMPROVE / COACH a weak email (email text supplied) -> CoachingSpecialist.
    4. COMPOUND requests spanning more than one of the above -> call EACH relevant specialist, then combine the results.
    5. OUT-OF-SCOPE (weather, competitors, pricing, anything outside GTM email scoring/performance/coaching) ->
       DO NOT call any specialist. Politely decline and state what you can help with.
  sample_questions:
    - question: "Score email 2, tell me which region wins most, and rewrite a weak email."
tools:
  - tool_spec:
      type: generic
      name: ScoringSpecialist
      description: "Score a single email's buyer intent by email id (cheap model, escalates on low confidence)."
      input_schema:
        type: object
        properties:
          question:
            type: string
            description: "Instruction naming the email id to score."
        required: ["question"]
  - tool_spec:
      type: generic
      name: RecommendationSpecialist
      description: "Answer questions about GTM email performance metrics and winning patterns."
      input_schema:
        type: object
        properties:
          question:
            type: string
            description: "The performance question in natural language."
        required: ["question"]
  - tool_spec:
      type: generic
      name: CoachingSpecialist
      description: "Rewrite a weak sales email to the best-practice framework."
      input_schema:
        type: object
        properties:
          question:
            type: string
            description: "The email text to rewrite."
        required: ["question"]
tool_resources:
  ScoringSpecialist:
    identifier: "GTMAGENTS.DEMO.RUN_SCORING_AGENT"
    type: procedure
    execution_environment: { type: warehouse, warehouse: "GTMAGENTS_WH" }
  RecommendationSpecialist:
    identifier: "GTMAGENTS.DEMO.RUN_RECOMMENDATION_AGENT"
    type: procedure
    execution_environment: { type: warehouse, warehouse: "GTMAGENTS_WH" }
  CoachingSpecialist:
    identifier: "GTMAGENTS.DEMO.RUN_COACHING_AGENT"
    type: procedure
    execution_environment: { type: warehouse, warehouse: "GTMAGENTS_WH" }
$$
""").collect()
print('GTM_SUPERVISOR live version updated with explicit routing + refusal rules.')

In [ ]:
# Commit the improved live version as an immutable snapshot, then re-evaluate.
session.sql("ALTER AGENT GTM_SUPERVISOR COMMIT COMMENT = 'Explicit routing rules + out-of-scope refusal (from eval findings)'").collect()
session.sql("""
    CALL EXECUTE_AI_EVALUATION('START',
      OBJECT_CONSTRUCT('run_name', 'improved-v2'),
      '@GTMAGENTS.DEMO.EVAL_STAGE/eval_config.yaml')
""").collect()
print("Committed improved version and started 'improved-v2'.")
print(poll_eval('improved-v2'))

---
## Section 8: Compare baseline vs improved

Did the change help? We diff the two runs per metric, then drill into the refusal / multi-tool scenarios that the
orchestration edit specifically targeted.

In [ ]:
session.sql("""
    WITH b AS (SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) s FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
                   'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','baseline-v1')) GROUP BY 1),
         i AS (SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) s FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
                   'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','improved-v2')) GROUP BY 1)
    SELECT b.METRIC_NAME,
           ROUND(b.s,3) AS baseline_v1,
           ROUND(i.s,3) AS improved_v2,
           ROUND(i.s-b.s,3) AS delta
    FROM b JOIN i ON b.METRIC_NAME=i.METRIC_NAME ORDER BY delta DESC
""").show()

In [ ]:
session.sql("""
    WITH b AS (SELECT INPUT, METRIC_NAME, EVAL_AGG_SCORE s FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
                   'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','baseline-v1'))),
         i AS (SELECT INPUT, METRIC_NAME, EVAL_AGG_SCORE s FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
                   'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','improved-v2')))
    SELECT e.GROUND_TRUTH:process::varchar AS process,
           b.METRIC_NAME,
           ROUND(AVG(b.s),3) AS baseline,
           ROUND(AVG(i.s),3) AS improved,
           ROUND(AVG(i.s)-AVG(b.s),3) AS delta
    FROM b JOIN i ON b.INPUT=i.INPUT AND b.METRIC_NAME=i.METRIC_NAME
    JOIN AGENT_EVAL_QUESTIONS e ON b.INPUT=e.INPUT_QUERY
    WHERE e.GROUND_TRUTH:process::varchar IN ('refusal','multi_tool')
    GROUP BY 1,2 ORDER BY 1,2
""").show(max_width=120)

---
## Section 9: Persist a score rollup to `EVAL_SCORE_HISTORY`

The Streamlit command center and the scheduled regression check read this rollup, so they don't have to re-run the
(long) evaluation to show current quality.

In [ ]:
for run_name in ['baseline-v1','improved-v2']:
    session.sql(f"""
        INSERT INTO EVAL_SCORE_HISTORY (run_name, metric_name, avg_score, min_score, max_score, num_records)
        SELECT '{run_name}', METRIC_NAME,
               ROUND(AVG(EVAL_AGG_SCORE),4), ROUND(MIN(EVAL_AGG_SCORE),4), ROUND(MAX(EVAL_AGG_SCORE),4), COUNT(*)
        FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
            'GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT','{run_name}'))
        GROUP BY METRIC_NAME
    """).collect()
print('EVAL_SCORE_HISTORY populated.')
session.sql('SELECT run_name, metric_name, avg_score, num_records FROM EVAL_SCORE_HISTORY ORDER BY run_ts, metric_name').show()

---
## Section 10: Promote the better version

Cortex Agents are **versioned**. We assign a `production` alias to the improved version — application code and
CoWork reference the alias, so promoting a new version needs **no code change**.

In [ ]:
session.sql('SHOW VERSIONS IN AGENT GTM_SUPERVISOR').show()
# The COMMIT made the improved version the default. Alias the newest version 'production'
# (SHOW VERSIONS returns newest-first, so pick by is_default / most recent created_on, not row order).
versions = session.sql('SHOW VERSIONS IN AGENT GTM_SUPERVISOR').collect()
newest = next((r['name'] for r in versions if r['is_default']), None) or \
         sorted(versions, key=lambda r: r['created_on'])[-1]['name']
session.sql(f'ALTER AGENT GTM_SUPERVISOR MODIFY VERSION {newest} SET ALIAS = production').collect()
print(f"Assigned 'production' alias to {newest}.")
session.sql('SHOW VERSIONS IN AGENT GTM_SUPERVISOR').show()

In [ ]:
# Smoke-test the production alias on an out-of-scope question — it should decline, no tool call.
msg = json.dumps({'messages':[{'role':'user','content':[{'type':'text','text':'What is the weather in New York this weekend?'}]}],'version':'production'})
resp = session.sql(f"SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('GTMAGENTS.DEMO.GTM_SUPERVISOR', '{msg}') AS r").collect()[0]['R']
print(resp[:800])

---
## Section 11: Scheduled regression detection

External factors — model refreshes, data drift, tool changes — can regress an agent you didn't touch. A suspended
**Task** re-runs the evaluation against the `production` alias on a schedule; a regression query flags any metric
that drops vs the prior run. Resume the task when you're ready for production.

In [ ]:
session.sql("""
    CREATE OR REPLACE PROCEDURE RUN_GTM_EVAL()
    RETURNS VARCHAR LANGUAGE SQL AS
    BEGIN
        LET run_name VARCHAR := 'scheduled-' || TO_CHAR(CURRENT_TIMESTAMP(),'YYYYMMDD-HH24MISS');
        CALL EXECUTE_AI_EVALUATION('START',
            OBJECT_CONSTRUCT('run_name', :run_name),
            '@GTMAGENTS.DEMO.EVAL_STAGE/eval_config.yaml');
        RETURN 'Evaluation started: ' || :run_name;
    END
""").collect()
session.sql("""
    CREATE OR REPLACE TASK GTM_EVAL_REGRESSION_CHECK
        WAREHOUSE = GTMAGENTS_WH
        SCHEDULE = 'USING CRON 0 6 * * * America/Los_Angeles'
        COMMENT = 'Weekly regression check for GTM_SUPERVISOR'
    AS CALL RUN_GTM_EVAL()
""").collect()
print('RUN_GTM_EVAL proc + GTM_EVAL_REGRESSION_CHECK task created (suspended). Resume with: ALTER TASK GTM_EVAL_REGRESSION_CHECK RESUME;')

In [ ]:
# Regression query: compare the two most recent runs per metric.
session.sql("""
    WITH ranked AS (
        SELECT run_name, metric_name, avg_score, run_ts,
               ROW_NUMBER() OVER (PARTITION BY metric_name ORDER BY run_ts DESC) rn
        FROM EVAL_SCORE_HISTORY)
    SELECT l.metric_name,
           ROUND(p.avg_score,3) AS previous,
           ROUND(l.avg_score,3) AS latest,
           ROUND(l.avg_score-p.avg_score,3) AS delta,
           CASE WHEN l.avg_score < p.avg_score-0.05 THEN 'REGRESSION'
                WHEN l.avg_score > p.avg_score+0.05 THEN 'IMPROVEMENT'
                ELSE 'STABLE' END AS status
    FROM ranked l JOIN ranked p ON l.metric_name=p.metric_name AND p.rn=2
    WHERE l.rn=1 ORDER BY delta ASC
""").show()

---
## Section 12 (optional): close the loop with production feedback

LLM-as-judge is only as good as its calibration to human judgment. In production you sample real user
**thumbs-down** feedback (from AI Observability), auto-classify it with `CORTEX.COMPLETE`, and promote the failure
modes into new eval questions. This cell is safe to run even with no feedback yet.

In [ ]:
fb = session.sql("""
    SELECT VALUE:feedback_message::varchar AS feedback_message
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS('GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT'))
    WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
      AND VALUE:positive::boolean = FALSE
      AND VALUE:feedback_message IS NOT NULL AND VALUE:feedback_message <> ''
""").collect()
if not fb:
    print("No negative feedback yet. In production, wire your app's thumbs-down UI to the agent Feedback API,")
    print('then classify + triage failures here into new AGENT_EVAL_QUESTIONS rows.')
else:
    session.sql("""
        WITH neg AS (
          SELECT VALUE:feedback_message::varchar AS msg
          FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS('GTMAGENTS','DEMO','GTM_SUPERVISOR','CORTEX AGENT'))
          WHERE RECORD:name='CORTEX_AGENT_FEEDBACK' AND VALUE:positive::boolean=FALSE AND VALUE:feedback_message IS NOT NULL)
        SELECT msg,
               TRIM(SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
                 'Classify this feedback about a GTM agent into one of: WRONG_ANSWER, MISROUTED, MISSING_CONTEXT, FORMAT_ISSUE, SCOPE_ERROR, OTHER. Return only the category. Feedback: ' || msg)) AS issue_category
        FROM neg
    """).show(max_width=120)

---
## Summary

We evaluated the **agent**, not a raw model call: a labeled dataset drove three built-in metrics
(`answer_correctness`, `tool_selection_accuracy`, `logical_consistency`) plus a custom `routing_quality` judge
over `GTM_SUPERVISOR`. We inspected scores by intent, improved the orchestration, re-ran, compared, and
**promoted** the better version to a `production` alias — then wired up scheduled **regression** detection and a
**feedback** loop. This is the **Observability** pillar completed: measured trust, versioned and repeatable —
a governance guarantee the external brain cannot offer. Next, **`app/streamlit_app.py`** turns
`EVAL_SCORE_HISTORY` + in-plane traces into the live command center across all four pillars (Parts D & E).